> 在线讲义：[第 21 章 · 存储层次与数据搬移：放得下、取得到、来得及](https://wikieden.github.io/ai-infra-from-constraints/lessons/21-memory-movement.html) · 本书《从约束推导 AI Infra》

# 第 21 章 · 存储层次与数据搬移：放得下、取得到、来得及

配套讲义：仓库里 `ai-infra/notes/lessons/21-memory-movement.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 工作台 | 输入 + 累加器 | `tile_kib(128, 128, 64)` | 96 KiB；双缓冲 128 KiB |
| 第二步 例 4-2 | Little 定律 | `inflight_needed(1.008e12, 500e-9)`、`effective_bw(...)` | 3,938；32.8 GB/s、36.9 ms；1.20 / 1.15 ms |
| 第三步 子矩阵 | 步长 | `submatrix_span(128, 128, 4096)` | 32 KiB 有效，跨度 1,040,640 B |
| 第四步 双缓冲 | 流水公式 | `pipe_time(8, 2, 1)` | 24 vs 17 μs |
| 第四步 缓冲槽 | 逐块模拟 | `simulate_slots(slots, t_c)` | 1280 / 768 / 704 / 704；×2 时 512 / 448 |

乘加块、例 4-2、子矩阵寻址、双缓冲、B200 单 SM 的输入槽时序取自 AII 第 4 章 §4.3–§4.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `tile_kib`**：一套乘加块的 (输入 + 权重 KiB, 累加器 KiB)。

In [ ]:
def tile_kib(m: int, n: int, k: int, b_in: int = 2, b_acc: int = 4) -> tuple[float, float]:
    """一套乘加块的 (输入 + 权重 KiB, 累加器 KiB)。"""
    return (m * k + k * n) * b_in / 1024, m * n * b_acc / 1024

**函数 `inflight_needed`**：维持目标带宽需要的在途事务数 ⌈R·L/s⌉。

In [ ]:
def inflight_needed(rate: float, latency_s: float, s: int = 128) -> int:
    """维持目标带宽需要的在途事务数 ⌈R·L/s⌉。"""
    return math.ceil(rate * latency_s / s)

**函数 `effective_bw`**：有效带宽 = min(接口带宽, N_o·s/L)。

In [ ]:
def effective_bw(n_o: int, latency_s: float, interface: float, s: int = 128) -> float:
    """有效带宽 = min(接口带宽, N_o·s/L)。"""
    return min(interface, n_o * s / latency_s)

**函数 `submatrix_span`**：从行宽 row_width 的矩阵取 rows×cols：(有效字节, 行步长, 首行起点到末行终点的跨度)。

In [ ]:
def submatrix_span(rows: int, cols: int, row_width: int, b: int = 2) -> tuple[int, int, int]:
    """从行宽 row_width 的矩阵取 rows×cols：(有效字节, 行步长, 首行起点到末行终点的跨度)。"""
    stride = row_width * b
    return rows * cols * b, stride, (rows - 1) * stride + cols * b

**函数 `pipe_time`**：(串行, 双缓冲流水) 的总时间。

In [ ]:
def pipe_time(n: int, t_l: float, t_c: float) -> tuple[float, float]:
    """(串行, 双缓冲流水) 的总时间。"""
    return n * (t_l + t_c), t_l + t_c + (n - 1) * max(t_l, t_c)

**函数 `simulate_slots`**：输入槽逐块模拟：传输串行，发起后 xfer+wait 就绪，算完才释放槽。返回完成时刻（tick）。

In [ ]:
def simulate_slots(slots: int, t_c: int, n: int = 4, xfer: int = 64, wait: int = 128) -> int:
    """输入槽逐块模拟：传输串行，发起后 xfer+wait 就绪，算完才释放槽。返回完成时刻（tick）。"""
    free_at = [0] * slots
    issue_free = 0
    comp_end = 0
    for _ in range(n):
        s = min(range(slots), key=lambda i: free_at[i])  # 最早空出来的槽
        issue = max(free_at[s], issue_free)
        issue_free = issue + xfer  # 传输串行
        start = max(issue + xfer + wait, comp_end)  # 就绪了、前一块也算完了
        comp_end = start + t_c
        free_at[s] = comp_end  # 算完才释放槽
    return comp_end

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：m = n = 128、k = 64 的乘加块")
inp, acc = tile_kib(128, 128, 64)
print(f"  输入 + 权重 {inp:.0f} KiB，累加器 {acc:.0f} KiB，合计 {inp + acc:.0f} KiB；双缓冲 {2 * inp + acc:.0f} KiB")

In [ ]:
print("\n第二步：例 4-2，读 1.125 GiB 的 KV")
kv = 1.125 * 2**30
print(f"  用满 1,008 GB/s 需要 {inflight_needed(1.008e12, 500e-9)} 个在途事务")
for n_o, iface in ((128, 1.008e12), (4096, 1.008e12), (4096, 1.792e12)):
    bw = effective_bw(n_o, 500e-9, iface)
    print(f"  {n_o:5d} 个，接口 {iface / 1e9:,.0f} GB/s → {bw / 1e9:,.1f} GB/s，{kv / bw * 1e3:.2f} ms")
print(f"  用满 1,792 GB/s：{inflight_needed(1.792e12, 500e-9)} 个；延迟 800 ns：{inflight_needed(1.792e12, 800e-9)} 个")

In [ ]:
print("\n第三步：从行宽 4096 的 BF16 矩阵取 128×128")
valid, stride, span = submatrix_span(128, 128, 4096)
print(f"  有效 {valid // 1024} KiB，行步长 {stride} B，跨度 {span:,} B")

In [ ]:
print("\n第四步：双缓冲与输入槽")
print(f"  8 块、加载 2 μs、计算 1 μs：串行 {pipe_time(8, 2, 1)[0]:.0f} μs，流水 {pipe_time(8, 2, 1)[1]:.0f} μs")
print(f"  输入槽 1–4：{[simulate_slots(s, 128) for s in (1, 2, 3, 4)]} tick；最早 {192 + 4 * 128}")
print(f"  矩阵 ×2：三槽 {simulate_slots(3, 64)}，四槽 {simulate_slots(4, 64)}（目标 {192 + 4 * 64}）")

## 3. 代码里才看得到的

1. 模拟里传输是串行的（同一时刻只有一块在传，`issue_free`），这是 64 tick 一块的前提；去掉这条，两个槽会算出更早的结果。
2. 槽在"这块算完"时才释放，不是数据到达时；这正是原文说槽"从发出读取请求到该块计算结束一直被占用"。
3. 例 4-2 的 1.125 GiB 是 $2^{30}$ 字节制，带宽是 $10^9$ 制；时间 $= 1.125 \times 2^{30} / R$。

## 4. 自己改一改

- 把 `simulate_slots` 的等待从 128 改成 256，三个槽还能连续计算吗？
- 用 `effective_bw` 扫在途事务 256 到 16384，画出图 21-2 的曲线。
- 把 `tile_kib` 的累加器移到独立存储（不计入），看 100 KB 里能放几对输入。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 工作台放两块板一块成品，备下一对还要再大；
> 数据、请求、流量三个数，配各自那一级的带宽；
> 服务员少了，厨房再快也送不出去；
> 小表格跳着抄，描述符替你算地址；
> 盆多了菜能边洗边炒，灶台不停就别再加盆；
> 队里交棒快，跨队先放桌上再喊话。